# 🎬 AI Scene Generator - AnimateDiff

Generate full animated scenes with character movement, emotions, and lip-sync.

**Instructions:**
1. Click **Runtime → Run all**
2. Upload your character image
3. Upload your audio file
4. Enter a description prompt
5. Wait for generation (10-30 minutes)
6. Download the result

**Free GPU:** This uses Google Colab's free GPU. Processing takes time but costs nothing!

In [ ]:
# Install dependencies
!pip install -q diffusers transformers accelerate xformers einops omegaconf safetensors
!pip install -q imageio imageio-ffmpeg moviepy
print("✓ Dependencies installed")

In [ ]:
# Download AnimateDiff models
from huggingface_hub import hf_hub_download
import os

os.makedirs("models", exist_ok=True)

print("Downloading AnimateDiff motion module...")
motion_module = hf_hub_download(
    repo_id="guoyww/animatediff-motion-adapter-v1-5-2",
    filename="mm_sd_v15_v2.ckpt",
    local_dir="models"
)

print("✓ Models downloaded")

In [ ]:
# Upload files
from google.colab import files
import shutil

print("📸 Upload your CHARACTER IMAGE:")
uploaded = files.upload()
character_image = list(uploaded.keys())[0]
print(f"✓ Character image: {character_image}")

print("\n🎵 Upload your AUDIO FILE:")
uploaded = files.upload()
audio_file = list(uploaded.keys())[0]
print(f"✓ Audio file: {audio_file}")

In [ ]:
# Get user prompt
prompt = input("\n✍️ Describe your scene (e.g., 'animated skeleton character speaking with emotions, dramatic lighting'): ")
print(f"\nPrompt: {prompt}")

In [ ]:
# Generate video
import torch
from diffusers import AnimateDiffPipeline, MotionAdapter, EulerDiscreteScheduler
from diffusers.utils import export_to_video
from PIL import Image
import numpy as np

print("🎬 Initializing AnimateDiff...")

# Load motion adapter
adapter = MotionAdapter.from_pretrained("guoyww/animatediff-motion-adapter-v1-5-2", torch_dtype=torch.float16)

# Load pipeline
pipe = AnimateDiffPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    motion_adapter=adapter,
    torch_dtype=torch.float16
).to("cuda")

pipe.scheduler = EulerDiscreteScheduler.from_config(pipe.scheduler.config)
pipe.enable_xformers_memory_efficient_attention()

print("✓ Pipeline ready")

# Load character image as reference
character = Image.open(character_image).convert("RGB")
character = character.resize((512, 512))

print("\n🎨 Generating animated scene...")
print("This will take 10-30 minutes. Please be patient!")

# Generate
output = pipe(
    prompt=prompt,
    num_frames=16,
    guidance_scale=7.5,
    num_inference_steps=25,
    generator=torch.Generator("cuda").manual_seed(42)
)

frames = output.frames[0]
export_to_video(frames, "animated_scene.mp4", fps=8)

print("✓ Animation generated!")

In [ ]:
# Add audio to video
from moviepy.editor import VideoFileClip, AudioFileClip

print("🎵 Adding audio to video...")

video = VideoFileClip("animated_scene.mp4")
audio = AudioFileClip(audio_file)

# Trim audio to match video length
if audio.duration > video.duration:
    audio = audio.subclip(0, video.duration)

final_video = video.set_audio(audio)
final_video.write_videofile("final_scene.mp4", codec="libx264", audio_codec="aac")

video.close()
audio.close()

print("✓ Final video ready!")

In [ ]:
# Download result
from google.colab import files

print("📥 Downloading your animated scene...")
files.download("final_scene.mp4")

print("\n✅ DONE! Your AI-generated scene is ready!")
print("\nYou can generate more scenes by running this notebook again.")